# PyTorch: Neural Networks

**Interview focus:** `nn.Module`, layers, activations, loss functions, and model architecture patterns.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## 1. Building Blocks

In [ ]:
# Linear layer: y = xW^T + b
linear = nn.Linear(in_features=10, out_features=5)
x = torch.randn(32, 10)  # batch of 32
out = linear(x)
print(f"out shape: {out.shape}")
print(f"weight shape: {linear.weight.shape}")  # (5, 10)
print(f"bias shape: {linear.bias.shape}")        # (5,)

In [ ]:
# Activations
x = torch.linspace(-3, 3, 7)
print(f"ReLU:  {F.relu(x)}")
print(f"Sigmoid: {torch.sigmoid(x).round(decimals=2)}")
print(f"Tanh:  {torch.tanh(x).round(decimals=2)}")
print(f"GELU:  {F.gelu(x).round(decimals=2)}")

## 2. Defining a Model with `nn.Module`

In [ ]:
class MLP(nn.Module):
  def __init__(self, input_dim, hidden_dim, output_dim):
    super().__init__()
    self.fc1 = nn.Linear(input_dim, hidden_dim)
    self.fc2 = nn.Linear(hidden_dim, hidden_dim)
    self.fc3 = nn.Linear(hidden_dim, output_dim)
    self.dropout = nn.Dropout(0.1)

  def forward(self, x):
    x = F.relu(self.fc1(x))
    x = self.dropout(x)
    x = F.relu(self.fc2(x))
    x = self.fc3(x)
    return x

model = MLP(784, 256, 10)
x = torch.randn(64, 784)
logits = model(x)
print(f"logits shape: {logits.shape}")
print(f"total params: {sum(p.numel() for p in model.parameters()):,}")

## 3. Sequential API

In [ ]:
model_seq = nn.Sequential(
  nn.Linear(784, 256),
  nn.ReLU(),
  nn.Dropout(0.1),
  nn.Linear(256, 10),
)

print(model_seq)

## 4. Loss Functions

In [ ]:
# Cross-entropy (classification) — expects raw logits, NOT softmax output
criterion = nn.CrossEntropyLoss()
logits = torch.randn(4, 3)       # 4 samples, 3 classes
targets = torch.tensor([0, 2, 1, 0])
loss = criterion(logits, targets)
print(f"CE loss: {loss.item():.4f}")

# MSE (regression)
mse = nn.MSELoss()
pred = torch.randn(4, 1)
target = torch.randn(4, 1)
print(f"MSE loss: {mse(pred, target).item():.4f}")

# Binary cross-entropy with logits
bce = nn.BCEWithLogitsLoss()
logits = torch.randn(4)
labels = torch.tensor([1.0, 0.0, 1.0, 0.0])
print(f"BCE loss: {bce(logits, labels).item():.4f}")

## 5. Convolutional Layer (for vision)

In [ ]:
conv = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
x = torch.randn(1, 3, 32, 32)  # (batch, channels, H, W)
out = conv(x)
print(f"conv out: {out.shape}")  # (1, 16, 32, 32) with padding=1

pool = nn.MaxPool2d(kernel_size=2, stride=2)
pooled = pool(out)
print(f"pooled: {pooled.shape}")  # (1, 16, 16, 16)

## 6. Batch Normalization

In [ ]:
bn = nn.BatchNorm1d(num_features=10)
x = torch.randn(32, 10)

bn.train()
out_train = bn(x)  # uses batch statistics

bn.eval()
out_eval = bn(x)   # uses running statistics

print(f"running mean: {bn.running_mean[:3]}")

## 7. Practice: Implement Attention Scores

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
  """Q, K, V: (batch, seq_len, d_k)"""
  d_k = Q.size(-1)
  scores = Q @ K.transpose(-2, -1) / (d_k ** 0.5)  # (batch, seq, seq)
  if mask is not None:
    scores = scores.masked_fill(mask == 0, float('-inf'))
  attn_weights = F.softmax(scores, dim=-1)
  return attn_weights @ V, attn_weights

batch, seq_len, d_k = 2, 4, 8
Q = torch.randn(batch, seq_len, d_k)
K = torch.randn(batch, seq_len, d_k)
V = torch.randn(batch, seq_len, d_k)

output, weights = scaled_dot_product_attention(Q, K, V)
print(f"output: {output.shape}, weights: {weights.shape}")
print(f"weights sum per row: {weights[0].sum(dim=-1)}")